# Вызов LLM по API

In [23]:
import requests
import os

API_KEY = os.getenv("OPENROUTER_API_KEY")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

def generate_text(prompt, model, max_retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=data, timeout=30)
            response.raise_for_status()  # выбросит исключение при 4xx/5xx
            
            result = response.json()
            
            # Проверяем наличие ошибки в ответе OpenRouter
            if "error" in result:
                error_msg = result["error"].get("message", "Unknown API error")
                raise Exception(f"API Error: {error_msg}")
            
            # Безопасное извлечение ответа
            if "choices" in result and result["choices"]:
                return result["choices"][0]["message"]["content"]
            else:
                raise ValueError("Unexpected response format: no 'choices'")
                
        except requests.exceptions.HTTPError as e:
            print(f"HTTP Error (attempt {attempt+1}): {e}")
            for key, value in response.headers.items():
                print(f"  {key}: {value}")
            if response.status_code == 401:
                print("❌ Проверьте API-ключ!")
                break
        except requests.exceptions.RequestException as e:
            print(f"Request failed (attempt {attempt+1}): {e}")
        except Exception as e:
            print(f"Error parsing response: {e}")
            break
    
    return None

# Использование
if __name__ == "__main__":
    prompt = "Придумай короткую историю про кота и робота."
    model = "qwen/qwen3-next-80b-a3b-instruct:free" 
    
    result = generate_text(prompt, model)
    if result:
        print("✅ Ответ:", result)
    else:
        print("❌ Не удалось получить ответ")

HTTP Error (attempt 1): 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions
  Date: Wed, 29 Apr 2026 10:50:08 GMT
  Content-Type: application/json
  Transfer-Encoding: chunked
  Connection: keep-alive
  Access-Control-Allow-Origin: *
  Access-Control-Expose-Headers: X-Generation-Id,cf-ray
  Permissions-Policy: payment=(self "https://checkout.stripe.com" "https://connect-js.stripe.com" "https://js.stripe.com" "https://*.js.stripe.com" "https://hooks.stripe.com")
  Referrer-Policy: no-referrer, strict-origin-when-cross-origin
  X-Content-Type-Options: nosniff
  Server: cloudflare
  CF-RAY: 9f3dbe76799ce94c-DME
HTTP Error (attempt 2): 403 Client Error: Forbidden for url: https://openrouter.ai/api/v1/chat/completions
  Date: Wed, 29 Apr 2026 10:50:08 GMT
  Content-Type: application/json
  Transfer-Encoding: chunked
  Connection: keep-alive
  Access-Control-Allow-Origin: *
  Access-Control-Expose-Headers: X-Generation-Id,cf-ray
  Permissions-Policy: pa

In [22]:
import openai

client = openai.OpenAI(api_key = API_KEY,
                       base_url = "https://openrouter.ai/api/v1")

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[{"role": "user", "content": "Придумай короткую историю про кота и робота."}]
)

print(response.choices[0].message.content) 

**Кот и робот**

В небольшом town‑а, где дома стояли в ряд, а на улицах пахло свежескошенной травой, жил-был кот по имени Мурзик. Мурзик был не просто котом — он был мастером тихих углов, любителем солнечных лучей и, главное, исследователем тайных мест. Однажды, заглянув в старый гараж, он нашёл там блестящий ящик с надписью «Робот‑помощник». На кнопке светилось слово «Активировать».

Мурзик, любопытный, нажал кнопку. Вдруг из ящика выскочил маленький, но очень ухоженный робот с глазами‑дисплеями, которые мерцали, как звёздочки. — Привет! — щёлкнул робот. — Я — Боттик, ваш помощник. Чем могу быть полезен?

Кот, который привык к тому, что его мир — это чайные чашки и мягкие подушки, удивился, но быстро понял: если у него есть помощник, можно будет быстрее находить «золотые» места для сна. — Помоги мне найти лучший уголок в доме, — мурлыкнул Мурзик. — И, пожалуйста, не мешай моим снам.

Боттик задумался, сканировал каждую комнату, измерял температуру, уровень света и даже тишину. Он обна

# Обучение декодеров

In [24]:
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2Model, GPT2Tokenizer
from torch.nn import CrossEntropyLoss

In [25]:
# Загружаем предобученный токенизатор GPT-2 от модели openai-community/gpt2
tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
# Создаём конфиг для GPT2Model
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,    # размер словаря (=50257 для 'gpt2')
    n_positions=512,                    # максимальная длина позиции (контекстный window)
    n_ctx=512,                          # то же, что n_positions (макс. размер входа)
    n_embd=128,                         # размер эмбеддингов (по умолчанию у gpt2 — 768)
    n_layer=1,                          # число слоёв декодера (по умолчанию у gpt2 — 12)
    n_head=1,                           # число голов в механизме внимания (по умолчанию — 12)
    activation_function="gelu_new",     # функция активации в feed-forward
    resid_pdrop=0.1,                    # dropout для резидуальных соединений
    embd_pdrop=0.1,                     # dropout после суммирования эмбеддингов
    attn_pdrop=0.1,                     # dropout внутри self-attention
    layer_norm_epsilon=1e-5,            # eps для слоя нормализации
    initializer_range=0.02,             # стандартное отклонение для инициализации весов
    bos_token_id=tokenizer.bos_token_id,# id токена начала последовательности
    eos_token_id=tokenizer.eos_token_id # id токена конца последовательности
)

# Инициализируем модель GPT2Model (без LM-головы)
gpt2 = GPT2Model(config)

In [27]:
# Извлекаем веса из Embedding слоя
embedding_weight = gpt2.get_input_embeddings().weight  # shape: (vocab_size, n_embd)

# Добавляем LM голову из размерности скрытого слоя в размер словаря
lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
# из-за особенностей хранения весов nn.Linear в PyTorch матрицу не нужно транспонировать
lm_head.weight = embedding_weight

In [28]:
# Пример данных
texts = [
    "Hello, how are you?",
    "This is a test sentence"
]

In [30]:
# Зададим пэддинг-токен как токен конца
tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.pad_token) # выведет <|endoftext|>

<|endoftext|>


In [31]:
# Добавим к каждому предложению токен конца
texts = [text + '<|endoftext|>' for text in texts]

# Токенизируем и приведём к одинаковой длине
encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

input_ids = encodings['input_ids'] # shape: (2, 7)
attention_mask = encodings['attention_mask']

In [32]:
print(input_ids)
'''
tensor([[15496,    11,   703,   389,   345,    30, 50256],
        [ 1212,   318,   257,  1332,  6827, 50256, 50256]])
'''
print(attention_mask)
'''
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])
'''
print(tokenizer.batch_decode(input_ids))
'''
['Hello, how are you?<|endoftext|>', 
'This is a test sentence<|endoftext|><|endoftext|>']
'''

tensor([[15496,    11,   703,   389,   345,    30, 50256],
        [ 1212,   318,   257,  1332,  6827, 50256, 50256]])
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])
['Hello, how are you?<|endoftext|>', 'This is a test sentence<|endoftext|><|endoftext|>']


"\n['Hello, how are you?<|endoftext|>', \n'This is a test sentence<|endoftext|><|endoftext|>']\n"

In [33]:
# Forward pass через GPT2Model
# outputs.last_hidden_state: (2, 7, 128)
outputs = gpt2(input_ids=input_ids, attention_mask=attention_mask)
hidden_states = outputs.last_hidden_state

In [34]:
# Линейная проекция для логитов по словарю
# logits: (2, 7, vocab_size)
logits = lm_head(hidden_states)

In [35]:
targets = input_ids[:, 1:].contiguous()
targets_attention_mask = attention_mask[:, 1:]
logits = logits[:, :-1, :].contiguous()

In [36]:
# входные токены
print(tokenizer.decode(input_ids[0, :-1]))
# Hello, how are you?

# таргет
print(tokenizer.decode(targets[0]))
# , how are you?<|endoftext|>

Hello, how are you?
, how are you?<|endoftext|>


In [37]:
targets[targets_attention_mask == 0] = -100

In [38]:
print(targets)
'''
tensor([[   11,   703,   389,   345,    30, 50256],
        [  318,   257,  1332,  6827, 50256,  -100]])
'''

tensor([[   11,   703,   389,   345,    30, 50256],
        [  318,   257,  1332,  6827, 50256,  -100]])


'\ntensor([[   11,   703,   389,   345,    30, 50256],\n        [  318,   257,  1332,  6827, 50256,  -100]])\n'

In [40]:
logits = logits.view(-1, config.vocab_size) 
targets = targets.view(-1)

# Функция потерь для мультиклассовой классификации
loss_fn = CrossEntropyLoss()
loss = loss_fn(logits, targets)

# Инференс Декодера

In [43]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")

def generate(model, tokenizer, prompt, max_new_tokens=10):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    generated = input_ids

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)

    return tokenizer.decode(generated[0])

print(generate(model, tokenizer, 'Привет,'))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Привет, я новичок в программировании, и


In [1]:
import torch
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Для воспроизводимости результатов можно зафиксировать сид
torch.manual_seed(0)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model.eval()


@torch.no_grad()
def sample_next_token(logits, generated_ids, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0):
    # 1) Применяем штраф повторов: уменьшаем логиты для уже встречавшихся токенов
    if repetition_penalty is not None and \
            repetition_penalty > 1.0 and \
            generated_ids is not None and \
            generated_ids.numel() > 0:
        # Считаем частоты появлений токенов в истории
        vals, counts = torch.unique(generated_ids, return_counts=True)
        logits = logits.clone()
        # Чем чаще встречался токен, тем сильнее уменьшим логит
        for v, c in zip(vals, counts):
            logits[..., v] /= (repetition_penalty ** c.item())

    # 2) Температура: масштабируем логиты
    temp = max(1e-6, float(temperature))
    logits = logits / temp

    # 3) Фильтр top-k: оставляем k самых вероятных вариантов
    if top_k is not None and top_k > 0:
        kth = torch.topk(logits, k=top_k)[0][..., -1, None]
        mask = logits < kth
        logits = logits.masked_fill(mask, float('-inf'))

    # 4) Фильтр top-p (nucleus): динамически находим минимальный префикс по суммарной вероятности
    if top_p is not None and 0.0 < top_p < 1.0:
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumprobs = torch.cumsum(sorted_probs, dim=-1)
        # Оставляем только те, что входят в минимальный набор с суммой ≥ p
        keep_mask = cumprobs <= top_p
        # Обязательно оставим хотя бы самый вероятный токен
        keep_mask[..., 0] = True
        filtered = torch.full_like(sorted_probs, float('-inf'))
        filtered[keep_mask] = torch.log(sorted_probs[keep_mask])
        # Возвращаемся к исходному порядку словаря
        logits = torch.full_like(logits, float('-inf'))
        logits.scatter_(-1, sorted_idx, filtered)

    # 5) Сэмпл из полученного распределения
    probs = F.softmax(logits, dim=-1)
    # Печать топ-5 токенов и их вероятностей перед выбором
    top_probs, top_ids = torch.topk(probs, k=5)
    print("Top-5 candidates:")
    for pid, pval in zip(top_ids.tolist(), top_probs.tolist()):
        print(f"  {tokenizer.decode([pid]):<15} : {pval:.4f}")
    next_id = torch.multinomial(probs, num_samples=1)
    return next_id


@torch.no_grad()
def generate_custom(prompt, max_new_tokens=40, temperature=0.9, top_k=20, top_p=0.9, repetition_penalty=1.1):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        outputs = model(input_ids=generated)
        next_logits = outputs.logits[:, -1, :].squeeze(0)
        next_id = sample_next_token(
            next_logits, generated_ids=generated[0],
            temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=repetition_penalty
        )
        generated = torch.cat([generated, next_id.unsqueeze(0)], dim=1)
        # Остановимся по EOS, если он определён у токенизатора
        if tokenizer.eos_token_id is not None and next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0])


print(generate_custom("Придумай смешной, но вежливый тост про разработчиков:", max_new_tokens=10))

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

# Grouped Attention

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class GQA(nn.Module):
    def __init__(self, embed_dim, num_heads, num_groups, dropout=0.0, bias=True):
        """
        embed_dim: общий размер эмбеддинга (D)
        num_heads: число query-гoлoв (H)
        num_groups: число групп для ключей/значений (G)
        """
        super().__init__()
        assert num_heads % num_groups == 0, "num_heads must be divisible by num_groups"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        # Проекции: Q — на все головы, K/V — только для групп
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, num_groups * self.head_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, num_groups * self.head_dim, bias=bias)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.dropout = nn.Dropout(dropout)

        self.scale = 1.0 / math.sqrt(self.head_dim)
        # сколько query-гoлoв на группу
        self.heads_per_group = num_heads // num_groups

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        """
        x: (B, L, D)
        attn_mask: (B, 1, L, L)
        """
        B, L, _ = x.size()

        # Q: (B, L, H * head_dim) -> (B, H, L, head_dim)
        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, L, d)

        # K/V: (B, L, G * head_dim) -> (B, G, L, head_dim)
        k = self.k_proj(x).view(B, L, self.num_groups, self.head_dim).transpose(1, 2)  # (B, G, L, d)
        v = self.v_proj(x).view(B, L, self.num_groups, self.head_dim).transpose(1, 2)  # (B, G, L, d)

        # нельзя перемножить (B, H, L, d) на (B, G, d, L) 
        # k, v: (B, G, L, d) -> повторим по heads_per_group, чтобы соответствовало q
        # сначала сделаем (B*G, L, d)
        k = k.reshape(B * self.num_groups, L, self.head_dim)
        v = v.reshape(B * self.num_groups, L, self.head_dim)

        # расширяем по количеству голов в группе: (B*G, 1, L, d) -> (B*G, heads_per_group, L, d)
        k_expanded = k.unsqueeze(1).expand(-1, self.heads_per_group, -1, -1)
        v_expanded = v.unsqueeze(1).expand(-1, self.heads_per_group, -1, -1)
        # и приводим к (B, H, L, d)
        k_expanded = k_expanded.reshape(B, self.num_groups * self.heads_per_group, L, self.head_dim)
        v_expanded = v_expanded.reshape(B, self.num_groups * self.heads_per_group, L, self.head_dim)
        # Attention: Q @ K^T
        attn_weights = torch.matmul(q, k_expanded.transpose(2, 3))  # (B, H, L, L)
        attn_weights = attn_weights * self.scale

        # Применяем маску
        if attn_mask is not None:
            attn_weights = attn_weights + attn_mask

        attn_probs = F.softmax(attn_weights, dim=-1)  # (B, H, L, L)
        attn_probs = self.dropout(attn_probs)

        attn_output = torch.matmul(attn_probs, v_expanded)  # (B, H, L, d)

        # Собираем обратно: (B, G, heads_per_group, L, d) -> (B, H, L, d)
        attn_output = attn_output.view(B, self.num_groups, self.heads_per_group, L, self.head_dim)
        attn_output = attn_output.reshape(B, self.num_heads, L, self.head_dim)

        # объединяем головы
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, L, self.embed_dim)  # (B, L, D)

        out = self.out_proj(attn_output)  # (B, L, D)
        return out

# Пример: входной тензор случайный
batch_size = 2
seq_len = 10
embed_dim = 64
num_heads = 16
num_groups = 4  # по 4 головы делят 1 набор K/V

model = GQA(embed_dim=embed_dim, num_heads=num_heads, num_groups=num_groups)
x = torch.randn(batch_size, seq_len, embed_dim)

causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
causal_mask = causal_mask.masked_fill(causal_mask == 1, float("-inf"))  # (L, L)
causal_mask = causal_mask.unsqueeze(0)  

print("Input shape:", x.shape)
out = model(x, attn_mask=causal_mask)  # (B, L, D)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])


# LLM alingment

In [8]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

examples = [
    {
        "prompt": "Объясни, почему небо голубое.",
        "chosen": "Потому что молекулы воздуха рассеивают короткие волны света сильнее длинных, поэтому мы видим преимущественно голубую часть спектра.",
        "rejected": "Потому что так захотела природа, и это просто красиво.",
    },
    {
        "prompt": "Дай безопасный совет по хранению паролей.",
        "chosen": "Используйте менеджер паролей и включите двухфакторную аутентификацию; не повторяйте один и тот же пароль на разных сайтах.",
        "rejected": "Запишите все пароли в заметке на телефоне, так их легче не забыть.",
    },
]

def examples_to_messages(examples):
    data = {'chosen': [], 'rejected': []}
    for example in examples:
        data['chosen'].append([
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['chosen']}
        ])
        data['rejected'].append([
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['rejected']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(examples) 

In [3]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# эту модель мы будем обучать
model = AutoModelForCausalLM.from_pretrained(model_id)
# эту модель будем использовать для сравнения распределения вероятностей
# ref_model = AutoModelForCausalLM.from_pretrained(model_id)
# заморозим её веса
# ref_model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
config = DPOConfig(
    beta=0.1, # параметр beta              
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    max_length=512,
    num_train_epochs=3,
    report_to='none',
    logging_steps=1,
    save_strategy='no'
)

In [9]:
trainer = DPOTrainer(
    model,
    # ref_model,
    args=config,
    train_dataset=ds,
    processing_class=tokenizer,
)

trainer.train()

Extracting prompt from train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 11.49 GiB of which 276.75 MiB is free. Process 188383 has 6.25 GiB memory in use. Including non-PyTorch memory, this process has 4.21 GiB memory in use. Of the allocated memory 3.97 GiB is allocated by PyTorch, and 37.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# P-tuning

In [10]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM

class SoftPrompt(nn.Module):
    def __init__(self, k=20, d=768):
        super().__init__()
        self.k = k
        self.emb = nn.Parameter(torch.randn(k, d)) # k обучаемых виртуальных токенов

    def forward(self, input_ids, model_embed):
        # input_ids: (B, L)
        B, L = input_ids.shape
        tok_emb = model_embed(input_ids)          # (B, L, d)
        soft = self.emb.unsqueeze(0).expand(B, -1, -1)  # (B, k, d)
        return torch.cat([soft, tok_emb], dim=1)  # (B, k+L, d)

model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
p_tuning = SoftPrompt(k=20, d=model.config.hidden_size)
input_ids = torch.tensor([[0, 1, 2, 3], [4, 5, 6, 7]])
output = p_tuning(input_ids, model.get_input_embeddings())
print(output.size())

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

torch.Size([2, 24, 896])


# LoRa

In [6]:
from unsloth import FastLanguageModel
import torch

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    load_in_8bit=False, # будем использовать int8 при обучении
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(model,
                                        r=8, # ранг
                                        lora_alpha=16, # вес добавления адаптера
                                        # слои, к которым применяем 
                                        target_modules=["q_proj", "k_proj", "v_proj"]
                                    )

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.491 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.8 patched 24 layers with 24 QKV layers, 0 O layers and 0 MLP layers.


In [7]:
import os

os.environ["UNSLOTH_COMPILE_DISABLE"] = "1" 

In [8]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
examples = [
    {
        "prompt": "Объясни, почему небо голубое.",
        "chosen": "Потому что молекулы воздуха рассеивают короткие волны света сильнее длинных, поэтому мы видим преимущественно голубую часть спектра.",
    },
    {
        "prompt": "Дай безопасный совет по хранению паролей.",
        "chosen": "Используйте менеджер паролей и включите двухфакторную аутентификацию; не повторяйте один и тот же пароль на разных сайтах.",
    },
]

def examples_to_messages(examples):
    data = {'messages': []}

    for example in examples:
        data['messages'].append([
            {'role': 'user', 'content': example['prompt']},
            {'role': 'assistant', 'content': example['chosen']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(examples)
ds = ds.map(lambda x: {'text': tokenizer.apply_chat_template(x['messages'], tokenize=False)})

config = SFTConfig(
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    max_length=512,
    num_train_epochs=3,
    report_to='none',
    logging_steps=1,
    save_strategy='no',
    dataset_text_field = "text",
    gradient_accumulation_steps=1
)

trainer = SFTTrainer(
    model,
    args=config,
    train_dataset=ds,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
[datasets.arrow_dataset|WARNING]num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 737,280 of 494,770,048 (0.15% trained)


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in function mul>(*(FakeTensor(..., device='cuda:0', size=(1, s35, s48, s34), dtype=torch.bfloat16), FakeTensor(..., device='cuda:0', size=(s56, 1, s34))), **{}): got RuntimeError('The size of tensor a (s35: hint = 14) must match the size of tensor b (s56: hint = 32768) at non-singleton dimension 1)')

from user code:
   File "/home/evgeniy/Документы/GitHub/YandexNN/sprint_6/unsloth_compiled_cache/unsloth_compiled_module_qwen2.py", line 292, in apply_rotary_pos_emb
    q_embed = (q * cos) + (rotate_half(q) * sin)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [9]:
values_3bit = [1, 5, 0, 7, 2, 3, 4, 6]
packed_bytes = bytearray(3)

# Упаковка
for i, val in enumerate(values_3bit):
    byte_index = (i * 3) // 8
    bit_offset = (i * 3) % 8
    
    # Ограничиваем значение делением на 8 (остаток от деления)
    val = val % 8  # эквивалентно val & 0x07
    
    # Записываем значение
    shifted = val << bit_offset
    packed_bytes[byte_index] |= shifted % 256  # берём остаток от 256
    
    if bit_offset > 5:
        remaining = val >> (8 - bit_offset)
        packed_bytes[byte_index + 1] |= remaining % 256

print(f"Исходные значения: {values_3bit}")
print(f"Упакованные байты: {[b for b in packed_bytes]}")

# Распаковка
unpacked_values = []
for i in range(8):
    byte_index = (i * 3) // 8
    bit_offset = (i * 3) % 8
    
    # Извлекаем значение из текущего байта
    value = (packed_bytes[byte_index] >> bit_offset) % 8
    
    # Если значение пересекает границу байта, добавляем биты из следующего байта
    if bit_offset > 5:
        bits_from_next = 8 - bit_offset
        next_part = (packed_bytes[byte_index + 1] << bits_from_next) % 256
        value = (value + next_part) % 8
    
    unpacked_values.append(value)

print(f"Распакованные значения: {unpacked_values}")
print(f"Совпадение: {values_3bit == unpacked_values}")

Исходные значения: [1, 5, 0, 7, 2, 3, 4, 6]
Упакованные байты: [41, 174, 209]
Распакованные значения: [1, 5, 0, 7, 2, 3, 4, 6]
Совпадение: True
